### Setup

In [37]:
!ls

05-deployment-homework.ipynb  model.py		   pyproject.toml      train.py
Dockerfile.txt		      model_metadata.json  python-version.txt  uv.lock
__pycache__		      pipeline.bin	   q6_test.py
feature_defaults.json	      predict.py	   smoke_test.py


In [38]:
!which python

/opt/conda/envs/ml-zoomcamp/bin/python


In [39]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


In [40]:
!uv sync --locked

Resolved 19 packages in 1ms
Checked 18 packages in 0.48ms


In [41]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


In [42]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


### Question 1 — environment check

In [43]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


### Question 2 — locked dependency

In [44]:
!grep -A 1 'scikit-learn==' pyproject.toml

    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",


In [45]:
!grep -A 1 'name = "scikit-learn"' uv.lock

    { name = "scikit-learn" },
    { name = "uvicorn" },
--
    { name = "scikit-learn", specifier = "==1.7.2" },
    { name = "uvicorn", specifier = "==0.38.0" },
--
name = "scikit-learn"
version = "1.7.2"


### Question 3 — load the model

In [46]:
lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

In [47]:
import pickle

with open("pipeline.bin", "rb") as f_in:
  pipeline = pickle.load(f_in)

proba = pipeline.predict_proba(lead)[0,1]
print(f"probability: {proba:.3f}")

probability: 0.533


### Question 4 — serve the model

In [22]:
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696

INFO:     Started server process [9370]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
^C
INFO:     Shutting down
INFO:     Finished server process [9370]


In [24]:
lead_2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

In [36]:
import requests

test_url = 'http://localhost:9696/health'
url = 'http://localhost:9696/predict'

test_response = requests.get(test_url).json()
if test_response['status']=='ok':
  response = requests.post(url, json=lead_2).json()
  assert response['conversion_probability'] > 0 and response['conversion_probability'] < 1
  assert isinstance(response['conversion'], bool)
  conversion_proba = response['conversion_probability']
  print(f"conversion probability: {conversion_proba:.3f}")

conversion probability: 0.770


### Question 5 — look at the container configuration

In [50]:
!uv pip install docker

Using Python 3.11.15 environment at: /opt/conda/envs/ml-zoomcamp
⠧ Resolving dependencies...                                                     error: Request failed after 3 retries in 48.1s
  cause: Failed to fetch: https://pypi.org/simple/docker/
  cause: error sending request for url (https://pypi.org/simple/docker/)
  cause: client error (Connect)
  cause: operation timed out


In [51]:
!docker build -t zoomcamp-model:2026-hw5 .

/bin/bash: line 1: docker: command not found
